# Reference trading price — WEM / SWIS

`ReferenceTradingPrice-*` carries one 30-minute series: the **reference trading price** in $/MWh,
the price that settles energy in the Wholesale Electricity Market. 51,168 intervals, no gaps and no
duplicates, from the market start on 1 October 2023 to 1 September 2026.

The obvious question to ask a price series is "what is the trend?", and the level *has* risen — but
the far larger change over these three years is that **the price stopped moving**. Volatility
collapsed, and the negative midday prices that dominated 2024 have all but disappeared. Those two
facts are the spine of this notebook, and they hand straight over to the battery analysis: the
daily price spread a battery earns its money from has fallen by roughly 80%.

Two things make cross-year comparison a trap here, and every chart below is built to survive them:

- **The administered cap changed.** It was $738/MWh through 2024 (binding on 166 intervals, last on
  11 December 2024) and rose to $1,100 from 1 January 2025. Some of the fall in "times at the cap"
  is a rule change, not a market outcome. The **floor** is different: prices reached exactly
  -$1,000 on 44 intervals in 2024, and by 2026 the annual minimum is -$16.72. Nothing moved to meet
  them; they simply stopped going there. Downside results compare at face value, upside ones need
  the cap stated.
- **2023 and 2026 are partial.** 2023 is October–December only (Spring and one month of Summer);
  2026 runs to 1 September, so it is missing November and December, which are the *cheap* months.
  Any full-year mean is therefore seasonally biased. Partial years are drawn dotted throughout,
  excluded from the seasonal grid outright, and the closing cell re-checks the headline on a
  like-for-like January–August basis.


In [1]:
# Load Reference Trading Price Data
import sys
sys.path.insert(0, '../src')  # Add src directory to path (go up one level from notebooks/)
from wa_data import load_price, add_time_parts

# Load with explicit path (go up one level from notebooks/)
df = load_price(raw='../data/raw')

In [2]:
print(df.head())
print(df.shape)
print(df.columns)

                   ts   price failure_reason
0 2023-10-01 08:00:00  555.99            NaN
1 2023-10-01 08:30:00  102.04            NaN
2 2023-10-01 09:00:00  143.80            NaN
3 2023-10-01 09:30:00    8.06            NaN
4 2023-10-01 10:00:00  132.61            NaN
(51168, 3)
Index(['ts', 'price', 'failure_reason'], dtype='object')


In [3]:
# Unlike demand and DPV, this series is ALREADY on the 30-minute trading interval,
# so there is no aggregation step: no `to_trading_intervals` call here. The
# `Extracted At` watermark is dropped by the loader; `failure_reason` is kept.
print(df.failure_reason.value_counts(dropna=False).to_string())

failure_reason
NaN                          50100
AffectedDispatchInterval       556
MarketAnalystOverride          445
DispatchEngineFailedToRun       51
MarketIsSuspended               16


In [4]:
# Completeness check. The whole record is used rather than a single YEAR, because
# the finding here is a change ACROSS years — but the file-boundary quirk still
# applies: these frames are the CONCATENATED files, and any per-year slice below
# is taken by local date, never from one file (each is cut at 00:00 UTC = 08:00
# AWST and so misses its own first 8 hours of January).
import pandas as pd

expected = int((df.ts.max() - df.ts.min()) / pd.Timedelta('30min')) + 1
print(f'coverage: {df.ts.min()}  ->  {df.ts.max()}')
print(f'30-min intervals: {len(df):,} of {expected:,} expected  ->  {expected - len(df)} missing')
print(f'duplicated timestamps: {df.ts.duplicated().sum()}')

coverage: 2023-10-01 08:00:00  ->  2026-09-01 07:30:00
30-min intervals: 51,168 of 51,168 expected  ->  0 missing
duplicated timestamps: 0


## Visualisation

Six views, ordered distribution -> time -> structure -> extremes, and landing on the handover to the
battery work:

1. **Price duration curves** — the whole distribution of each year in one line.
2. **Level and volatility over the record** — rolling median with a rolling 10th–90th band.
3. **The average day, by season and year** — where in the day the change happened.
4. **One week, 2024 against 2026** — the same calendar week, at full resolution.
5. **Negative prices carpet** — when they happened, and when they stopped.
6. **Daily spread against storage charging** — the arbitrage signal, and the likely cause.

Unlike the demand and DPV notebooks there is no single `YEAR` to set: the subject here is the
difference between years, so the year is a *series*, not a slice.


In [5]:
# ── Chart setup: palette, shared theme, and derived frames ───────────────────
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chart chrome, light surface — identical to the demand and DPV notebooks, so the
# three read as one system.
SURFACE, INK, INK_2, MUTED, GRID, AXIS = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7")

BLUE, ORANGE, RED = "#2a78d6", "#eb6834", "#e34948"

# The YEAR is this notebook's series, and years are ORDINAL, not categorical, so
# they take a one-hue ramp light->dark instead of four cycled hues: the later the
# year, the darker the line, which puts the direction of travel in the colour
# itself. Steps 250/400/500/700 of the reference blue ramp; validated as an
# ordinal ramp (monotone lightness, adjacent dL >= 0.06, light end 2.06:1 on this
# surface, hue spread 4 degrees). Colour follows the YEAR, never its rank, so
# 2024 keeps the same step in a chart that drops 2023.
YEAR_COLOR = {2023: "#86b6ef", 2024: "#3987e5", 2025: "#256abf", 2026: "#0d366b"}

# Single-hue sequential ramp for the carpet; orange stays "storage", as in the
# demand notebook, where withdrawal is charging.
RAMP_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

# Partial years are drawn dotted everywhere they appear: dash is the secondary
# encoding that carries "do not read this as a full year" without a second hue.
PARTIAL = {2023: "Oct–Dec only", 2026: "to 1 Sep"}
YEARS = [2023, 2024, 2025, 2026]
LABEL = {y: (f"{y} ({PARTIAL[y]})" if y in PARTIAL else str(y)) for y in YEARS}
DASH = {y: ("dot" if y in PARTIAL else "solid") for y in YEARS}


def style(fig, title, subtitle=None, height=420, hover="x unified",
          top=108, bottom=58, legend_y=1.0):
    """Shared theme: light surface, recessive grid, muted axes, ink-coloured text."""
    head = f"<b>{title}</b>"
    if subtitle:
        head += f"<br><span style='font-size:12.5px;color:{INK_2}'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=head, font=dict(size=17, color=INK), x=0, xanchor="left", y=0.96),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, hovermode=hover,
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=12, color=INK_2),
        height=height, margin=dict(t=top, r=30, b=bottom, l=74),
        legend=dict(orientation="h", yanchor="bottom", y=legend_y, x=0,
                    bgcolor="rgba(0,0,0,0)", font=dict(color=INK_2)),
    )
    fig.update_xaxes(showgrid=False, linecolor=AXIS, ticks="outside",
                     tickcolor=AXIS, tickfont=dict(color=MUTED))
    fig.update_yaxes(gridcolor=GRID, linecolor="rgba(0,0,0,0)", tickfont=dict(color=MUTED))
    return fig


# The whole record with calendar / time-of-day helpers attached.
d = add_time_parts(df.copy())

# Per-day aggregates, reused by charts 2 and 6. `spread` is the daily high minus
# the daily low: the most a perfectly-informed battery could earn per MWh cycled
# that day, before efficiency losses.
daily = (d.groupby("date")
         .agg(mean=("price", "mean"), median=("price", "median"),
              lo=("price", "min"), hi=("price", "max"))
         .reset_index())
daily["spread"] = daily.hi - daily.lo
daily["year"] = daily.date.dt.year

# Months holding a full complement of days. The record starts 1 Oct 2023 08:00
# and ends 1 Sep 2026 07:30, so its final month holds a SINGLE day. Left in, that
# month draws a cliff at the right edge of every monthly chart which is an
# artefact of where the extract stops, not a market event. Charts 5 and 6 use this.
mdays = d.groupby(d.ts.dt.to_period("M"))["date"].nunique()
FULL_MONTHS = set(mdays[mdays >= 20].index.to_timestamp())
print("partial months excluded from the monthly charts:",
      [str(i) for i in mdays[mdays < 20].index])

print(f"{len(d):,} intervals over {d.date.nunique():,} days")
print("\nprice by year ($/MWh):")
print(d.groupby("year")["price"].agg(
    n="size", mean="mean", median="median", std="std",
    min="min", max="max").round(1).to_string())

partial months excluded from the monthly charts: ['2026-09']
51,168 intervals over 1,067 days

price by year ($/MWh):
          n   mean  median    std     min     max
year                                             
2023   4400   83.8    73.6  134.6  -716.7   738.0
2024  17568   79.3    83.0  141.5 -1000.0   738.0
2025  17520   87.7    88.2   56.3  -150.5  1100.0
2026  11680  109.0   102.4   45.3   -16.7  1000.0


### 1. Price duration curves

Sort every interval of a year from most to least expensive and plot it against the share of the year:
the whole distribution becomes one line, and four years become four comparable lines. It is the
standard market view, and it answers "how much of the year was above $300?" by reading across.

The left panel keeps the full range so the tails are visible; the right panel zooms to the body,
where most of the year actually sits. Read the pair together — the tails shrink *and* the body lifts
and flattens. 2023 and 2026 are dotted: partial years, so their shape is seasonally biased.


In [6]:
# ── Chart 1: price duration curves ───────────────────────────────────────────
# NOTE: plotly runs titles through MathJax, so a string containing TWO dollar
# signs is parsed as LaTeX and rendered as gibberish. Any price range in chart
# text is therefore written with the unit outside, never as "$-50 to $300".
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=["Full range", "The body, -50 to 300 $/MWh"])

for y in YEARS:
    v = np.sort(d.loc[d.year == y, "price"].to_numpy())[::-1]
    x = np.linspace(0, 100, len(v))
    for col in (1, 2):
        fig.add_trace(go.Scatter(
            x=x, y=v, name=LABEL[y], legendgroup=str(y), showlegend=(col == 1),
            line=dict(color=YEAR_COLOR[y], width=2, dash=DASH[y]),
            hovertemplate="dearest %{x:.1f}% of intervals<br>$%{y:,.0f}/MWh"
                          f"<extra>{LABEL[y]}</extra>"), row=1, col=col)
    # Direct labels are SELECTIVE: only the two years the story runs between. All
    # four would collide, because the 2023 and 2024 curves are near-identical and
    # no x position separates them. The legend carries the other two. Label text
    # stays in ink, not the series colour — it sits on its own line, so position
    # carries identity.
    if y in (2024, 2026):
        fig.add_annotation(x=85, y=float(np.interp(85, x, v)), text=str(y),
                           showarrow=False, yshift=13,
                           font=dict(size=11, color=INK_2), row=1, col=2)

for col in (1, 2):
    fig.add_hline(y=0, line=dict(color=AXIS, width=1, dash="dot"), row=1, col=col)

style(fig, "Price duration curves — every interval of each year, sorted",
      "Share of the year at or above a given price. Later years are darker; "
      "dotted years are incomplete.", height=490, hover="closest",
      top=150, legend_y=1.14)
fig.update_xaxes(title_text="% of intervals", range=[0, 100], ticksuffix="%")
fig.update_yaxes(title_text="$/MWh", row=1, col=1)
fig.update_yaxes(range=[-50, 300], row=1, col=2)
fig.show()

print("share of intervals above $300/MWh, and below $0:")
print(d.groupby("year")["price"].agg(
    above_300=lambda s: f"{(s > 300).mean():.2%}",
    below_0=lambda s: f"{(s < 0).mean():.2%}").to_string())

share of intervals above $300/MWh, and below $0:
     above_300 below_0
year                  
2023     5.84%  20.57%
2024     4.65%  21.82%
2025     0.67%   5.30%
2026     0.61%   0.41%


### 2. Level and volatility over the record

The duration curves compare years but throw away time. This puts the record back on a calendar axis
and separates the two movements that summary statistics blend together: the **line** is the rolling
30-day median (the level), the **band** is the rolling 10th-to-90th percentile (the spread).

A rolling window is used rather than raw values because at 30-minute resolution the raw series is
unreadable hash. The first 30 days are blanked, since their window is not yet full.

The dashed marker is 1 January 2025, when the administered cap moved from $738 to $1,100. The band
does not step at that date: it is already narrowing through December 2024, and it keeps narrowing
for months afterwards. The monthly widths printed below make that checkable, and they are the
evidence that the calming is not merely the rule change.


In [7]:
# ── Chart 2: rolling level and spread ────────────────────────────────────────
WIN = "30D"
r = d.set_index("ts")["price"].sort_index()
roll = pd.DataFrame({
    "median": r.rolling(WIN).median(),
    "p10": r.rolling(WIN).quantile(0.10),
    "p90": r.rolling(WIN).quantile(0.90),
})
roll[roll.index < r.index.min() + pd.Timedelta(WIN)] = np.nan   # partial windows
roll = roll.resample("1D").last().dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=roll.index, y=roll.p90, line=dict(color="rgba(0,0,0,0)"),
    hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(
    x=roll.index, y=roll.p10, name="30-day 10th–90th percentile",
    line=dict(color="rgba(0,0,0,0)"), fill="tonexty",
    fillcolor="rgba(42,120,214,0.18)",
    hovertemplate="$%{y:,.0f}<extra>10th percentile</extra>"))
fig.add_trace(go.Scatter(
    x=roll.index, y=roll["median"], name="30-day median",
    line=dict(color=BLUE, width=2),
    hovertemplate="$%{y:,.0f}<extra>median</extra>"))

fig.add_hline(y=0, line=dict(color=AXIS, width=1, dash="dot"))
fig.add_vline(x=pd.Timestamp("2025-01-01"), line=dict(color=MUTED, width=1, dash="dash"))
fig.add_annotation(x=pd.Timestamp("2025-01-05"), y=1, yref="paper", yanchor="top",
                   text="cap 738 → 1,100 $/MWh", showarrow=False, xanchor="left",
                   font=dict(size=11, color=MUTED))

style(fig, "The price stopped moving before it started rising",
      "Rolling 30-day median (line) and 10th–90th percentile band, "
      "across the whole record.", height=430)
fig.update_yaxes(title_text="$/MWh")
fig.show()

w = roll.p90 - roll.p10
print("mean width of the 10th–90th band, by year ($/MWh):")
print(w.groupby(w.index.year).mean().round(0).to_string())

# Does the band STEP at the cap change, or narrow through it? Monthly widths
# either side settle it — a step would show as a break at January 2025.
mw = d.assign(ms=d.ts.dt.to_period("M").dt.to_timestamp()).groupby("ms")["price"]
mw = (mw.quantile(0.90) - mw.quantile(0.10)).round(0)
print("\nmonthly 10th–90th width either side of the cap change ($/MWh):")
print(mw.loc["2024-09":"2025-05"].to_string())

mean width of the 10th–90th band, by year ($/MWh):
ts
2023    216.0
2024    197.0
2025    101.0
2026     71.0

monthly 10th–90th width either side of the cap change ($/MWh):
ms
2024-09-01    194.0
2024-10-01    286.0
2024-11-01    189.0
2024-12-01    141.0
2025-01-01    107.0
2025-02-01    106.0
2025-03-01    119.0
2025-04-01     94.0
2025-05-01     82.0


### 3. The average day, by season and year

Averaging every day of a season onto one 24-hour axis turns thousands of intervals into 48 points per
line, and shows *where in the day* the change happened. It is not uniform, and it is not a downward
shift: the evening peak has come down hard (mean 18:00 price $204 -> $137 between 2024 and 2026)
while the overnight price has gone *up* ($70 -> $102 across 00:00–05:00). The daily shape is
flattening from both ends at once, which is the signature of storage arbitrage — buy the trough,
sell the peak, and both move toward each other.

Seasons are meteorological and southern-hemisphere (Summer is Dec–Feb), matching the other two
notebooks. A (year, season) pair is plotted only if **all three of its months are present** and it
clears `MIN_DAYS` — which drops 2023 entirely, plus Summer and Spring 2026. That is stricter than a
day count alone: Summer 2026 has 59 days and would pass a day threshold, but it is January and
February with no December, so it is not a season.


In [8]:
# ── Chart 3: average day by season and year ──────────────────────────────────
SEASON_ORDER = ["Summer", "Autumn", "Winter", "Spring"]
SEASON_MONTHS = {"Summer": {12, 1, 2}, "Autumn": {3, 4, 5},
                 "Winter": {6, 7, 8}, "Spring": {9, 10, 11}}
MIN_DAYS = 30

have = d.groupby(["year", "season"]).agg(days=("date", "nunique"),
                                         months=("month", lambda s: set(s.unique())))
keep = {(y, s) for (y, s), row in have.iterrows()
        if row.days >= MIN_DAYS and row.months == SEASON_MONTHS[s]}
print("plotted:", sorted(keep))
print("dropped:", sorted(set(have.index) - keep))

prof = d.groupby(["year", "season", "tod_min"])["price"].mean().reset_index()

fig = make_subplots(rows=1, cols=4, shared_yaxes=True, horizontal_spacing=0.025,
                    subplot_titles=SEASON_ORDER)
seen = set()
for i, s in enumerate(SEASON_ORDER, start=1):
    for y in YEARS:
        if (y, s) not in keep:
            continue
        g = prof[(prof.year == y) & (prof.season == s)].sort_values("tod_min")
        fig.add_trace(go.Scatter(
            x=g.tod_min / 60, y=g.price, name=LABEL[y], legendgroup=str(y),
            showlegend=y not in seen,
            line=dict(color=YEAR_COLOR[y], width=2, dash=DASH[y]),
            hovertemplate="%{x:.1f}h · $%{y:,.0f}/MWh" f"<extra>{LABEL[y]} {s}</extra>"),
            row=1, col=i)
        seen.add(y)
    fig.add_hline(y=0, line=dict(color=AXIS, width=1, dash="dot"), row=1, col=i)

style(fig, "The daily shape is flattening from both ends",
      "Mean price by time of day. Only seasons with all three months present are shown.",
      height=460, top=150, legend_y=1.14)
fig.update_xaxes(tickvals=[0, 6, 12, 18, 24], range=[0, 24], title_text="hour")
fig.update_yaxes(title_text="$/MWh", row=1, col=1)
fig.show()

pk = (d[d.year.isin([2024, 2025, 2026])]
      .groupby(["year", "hour"])["price"].mean().unstack(0).round(0))
print("mean price at the evening peak ($/MWh):")
print(pk.loc[[17, 18, 19]].to_string())
print("\nboth ends, 2024 -> 2026 ($/MWh):")
on = d[d.hour <= 5].groupby("year")["price"].mean().round(1)
pkh = d[d.hour == 18].groupby("year")["price"].mean().round(1)
for y in (2024, 2025, 2026):
    print(f"  {y}:  overnight 00:00–05:00 ${on[y]:>6.1f}   peak 18:00 ${pkh[y]:>6.1f}"
          f"   gap ${pkh[y] - on[y]:>6.1f}")

plotted: [(2024, 'Autumn'), (2024, 'Spring'), (2024, 'Summer'), (2024, 'Winter'), (2025, 'Autumn'), (2025, 'Spring'), (2025, 'Summer'), (2025, 'Winter'), (2026, 'Autumn'), (2026, 'Winter')]
dropped: [(2023, 'Spring'), (2023, 'Summer'), (2026, 'Spring'), (2026, 'Summer')]


mean price at the evening peak ($/MWh):
year   2024   2025   2026
hour                     
17    188.0  130.0  128.0
18    204.0  143.0  137.0
19    155.0  124.0  128.0

both ends, 2024 -> 2026 ($/MWh):
  2024:  overnight 00:00–05:00 $  70.1   peak 18:00 $ 203.5   gap $ 133.4
  2025:  overnight 00:00–05:00 $  84.1   peak 18:00 $ 143.1   gap $  59.0
  2026:  overnight 00:00–05:00 $ 102.3   peak 18:00 $ 136.8   gap $  34.5


### 4. One week, 2024 against 2026

The seasonal means above are smooth by construction, and smoothness is exactly what a trader does not
experience. This keeps every 30-minute interval and narrows the window to a single week, drawn twice:
the mid-April week of 2024 and of 2026, each starting on the first Monday on or after 15 April so
that weekday effects line up.

Mid-April is chosen because it is deep enough into autumn to be free of summer peaks while still
carrying strong midday solar — the conditions that produced negative prices in 2024. The red band is
the region below zero, where a generator pays to run.


In [9]:
# ── Chart 4: one week, two years, same calendar week ─────────────────────────
def monday_after(year, month=4, day=15):
    """First Monday on or after `day` — keeps weekday alignment across years."""
    t = pd.Timestamp(year=year, month=month, day=day)
    return t + pd.Timedelta(days=(7 - t.dayofweek) % 7)

fig = go.Figure()
lows = []
for y in (2024, 2026):
    start = monday_after(y)
    wk = d[(d.ts >= start) & (d.ts < start + pd.Timedelta(days=7))]
    lows.append(wk.price.min())
    fig.add_trace(go.Scatter(
        x=(wk.ts - start) / pd.Timedelta(hours=1), y=wk.price,
        name=f"{y} · week of {start.strftime('%d %b')}",
        line=dict(color=YEAR_COLOR[y], width=2),
        hovertemplate="$%{y:,.0f}/MWh" f"<extra>{y}</extra>"))

ymin = min(lows) * 1.15
fig.add_hrect(y0=ymin, y1=0, fillcolor="rgba(227,73,72,0.10)", line_width=0, layer="below")
fig.add_hline(y=0, line=dict(color=AXIS, width=1))

# The "below zero" note lives in the subtitle rather than as an in-plot
# annotation: anywhere inside the red band it would sit on top of the 2024 line,
# which spends much of the week there — which is the very point of the chart.
style(fig, "The same April week, two years apart",
      "Every 30-minute interval, aligned Monday to Monday. In the red band the "
      "price is below zero and generators pay to run.", height=430)
fig.update_xaxes(title_text=None, tickvals=list(range(0, 169, 24)),
                 ticktext=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun", ""],
                 range=[0, 168])
fig.update_yaxes(title_text="$/MWh")
fig.show()

### 5. When the negative prices happened — and when they stopped

This is the "extremes" question asked in a way that survives the rule change. Counting intervals at
the *cap* mostly measures where the cap was set; counting intervals **below zero** measures the
market, because the floor never moved.

One column per month across the whole record, one row per half hour, colour is the share of that
half hour's intervals that settled below $0. The solid midday block through 2024 is rooftop solar
pushing the middle of the day into oversupply. It fades over 2025 and is essentially gone by 2026 —
which is what a storage fleet learning to absorb the midday surplus looks like from the price side.


In [10]:
# ── Chart 5: negative-price carpet ───────────────────────────────────────────
def to_surface(ramp):
    """Anchor the ramp's zero end on the chart surface.

    Most of this carpet is zero — negative prices never occur at night in any
    year. Left alone the ramp's lightest step tints every zero cell and the plot
    spends ink on nothing; blending the low end into the surface lets the midday
    block draw itself.
    """
    stops = [SURFACE] + list(ramp)
    return [[i / (len(stops) - 1), c] for i, c in enumerate(stops)]


neg = d.assign(neg=(d.price < 0).astype(float),
               mstart=d.ts.dt.to_period("M").dt.to_timestamp())
neg = neg[neg.mstart.isin(FULL_MONTHS)]        # drop the 1-day final month
piv = (neg.pivot_table(index="tod_min", columns="mstart", values="neg", aggfunc="mean")
       .sort_index() * 100)

fig = go.Figure(go.Heatmap(
    z=piv.values, x=piv.columns, y=piv.index / 60,
    colorscale=to_surface(RAMP_BLUE), zmin=0, zmax=float(np.nanmax(piv.values)),
    zsmooth=False,
    colorbar=dict(title=dict(text="% of<br>intervals", side="top"), outlinewidth=0,
                  thickness=12, tickfont=dict(color=MUTED), ticksuffix="%"),
    hovertemplate="%{x|%b %Y} · %{y:.1f}h<br>%{z:.0f}% below $0<extra></extra>"))

style(fig, "Negative prices: a solid midday block in 2024, gone by 2026",
      "Share of each half hour's intervals settling below $0/MWh, by month.",
      height=420, hover="closest")
fig.update_yaxes(title_text="hour of day", tickvals=[0, 6, 12, 18, 24],
                 range=[0, 24], gridcolor="rgba(0,0,0,0)")
fig.update_xaxes(showgrid=False, tickformat="%b<br>%Y")
fig.show()

print("intervals below $0, by year:")
print(d.groupby("year")["price"].agg(
    n_neg=lambda s: int((s < 0).sum()),
    share=lambda s: f"{(s < 0).mean():.2%}",
    deepest=lambda s: round(s.min(), 2)).to_string())

intervals below $0, by year:
      n_neg   share  deepest
year                        
2023    905  20.57%  -716.71
2024   3833  21.82% -1000.00
2025    929   5.30%  -150.52
2026     48   0.41%   -16.72


### 6. The daily spread, and the storage fleet that ate it

The daily high minus the daily low is what a battery actually sells: charge at the low, discharge at
the high, and the spread is the gross margin per MWh cycled. Its median has fallen from $663 a day
in 2023 to $93 in 2026.

The lower panel is grid-scale storage charging, taken from the demand notebook's `withdrawal_mw` —
the scheduled-load series that the verified demand identity isolates as storage charging. It is
shown as a **second panel sharing the x-axis, never as a second y-axis on the same panel**: the two
measures have unrelated units, and overlaying them on twin scales would let the choice of scale
manufacture whatever correlation the reader was looking for.

The panels are aligned in time and left to speak for themselves. Storage capacity rising as spread
falls is consistent with batteries competing away the very arbitrage that justified building them,
but this notebook does not establish that. Two series moving oppositely over three years is weak
evidence, and demand growth, the cap change and gas costs all move in here too. The battery notebook
is where that gets tested properly.


In [11]:
# ── Chart 6: daily spread against storage charging ───────────────────────────
from wa_data import load_demand

dem = load_demand(raw='../data/raw')
stor = (dem.assign(mstart=dem.ts.dt.to_period("M").dt.to_timestamp())
        .groupby("mstart")["withdrawal_mw"].min().abs())   # withdrawal is <= 0
stor = stor[stor.index.isin(FULL_MONTHS)]

sp = daily.assign(mstart=daily.date.dt.to_period("M").dt.to_timestamp())
sp = sp[sp.mstart.isin(FULL_MONTHS)]           # both panels drop the 1-day final month
spm = sp.groupby("mstart")["spread"].agg(
    med="median", q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    row_heights=[0.62, 0.38],
                    subplot_titles=["Daily price spread (high minus low)",
                                    "Peak grid-scale storage charging in the month"])

fig.add_trace(go.Scatter(x=spm.index, y=spm.q3, line=dict(color="rgba(0,0,0,0)"),
                         hoverinfo="skip", showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=spm.index, y=spm.q1, line=dict(color="rgba(0,0,0,0)"),
                         fill="tonexty", fillcolor="rgba(42,120,214,0.18)",
                         name="interquartile range of daily spreads", legendrank=2,
                         hovertemplate="$%{y:,.0f}<extra>25th percentile</extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=spm.index, y=spm.med, line=dict(color=BLUE, width=2),
                         name="median daily spread", legendrank=1,
                         hovertemplate="$%{y:,.0f}<extra>median</extra>"), row=1, col=1)

fig.add_trace(go.Scatter(x=stor.index, y=stor.values, line=dict(color=ORANGE, width=2),
                         fill="tozeroy", fillcolor="rgba(235,104,52,0.14)",
                         name="peak storage charging", legendrank=3,
                         hovertemplate="%{y:,.0f} MW<extra>storage charging</extra>"),
              row=2, col=1)

style(fig, "The arbitrage spread has fallen by about 80%",
      "Monthly median of the daily high-minus-low, with its interquartile range. "
      "Separate panels, one shared time axis — never a second y-axis.",
      height=590, top=150, legend_y=1.10)
fig.update_yaxes(title_text="$/MWh", row=1, col=1)
fig.update_yaxes(title_text="MW", row=2, col=1)
fig.show()

print("daily spread ($/MWh), by year:")
print(daily.groupby("year")["spread"].agg(
    mean="mean", median="median", p90=lambda s: s.quantile(0.90)).round(0).to_string())

daily spread ($/MWh), by year:
       mean  median    p90
year                      
2023  611.0   663.0  807.0
2024  573.0   574.0  934.0
2025  194.0   148.0  351.0
2026  118.0    93.0  240.0


## Comparability checks

Three things could each have manufactured the story above on their own. Checking them is the
difference between a finding and an artefact.

1. **The partial years.** 2026 is missing November and December, the two cheapest months of 2025
   ($58 and $66 mean). If the rise in level were only that, a like-for-like January–August
   comparison would erase it.
2. **The cap change.** Counting intervals at the maximum across the boundary compares administered
   parameters, not markets. The floor is not affected.
3. **`MarketAnalystOverride`.** Manually set prices jump from 1 interval in 2024 to 237 in 2025 and
   207 in 2026. Any price result should be re-checked with them removed.


In [12]:
# ── Comparability checks ─────────────────────────────────────────────────────
ja = d[d.month <= 8]                      # like-for-like: every year has Jan–Aug complete
print("1. LIKE-FOR-LIKE (January–August only) — the rise survives:")
print(ja.groupby("year")["price"].agg(n="size", mean="mean", median="median",
                                      std="std").round(1).to_string())
print("\n   full-year vs Jan–Aug mean, for the years where both exist:")
for y in (2024, 2025):
    print(f"     {y}:  full ${d[d.year == y].price.mean():.1f}"
          f"   Jan–Aug ${ja[ja.year == y].price.mean():.1f}")

print("\n2. CAP AND FLOOR — 'times at the max' is partly a rule change; the floor is not:")
for y in YEARS:
    a = d[d.year == y]
    print(f"     {y}:  max ${a.price.max():>8,.2f} on {(a.price == a.price.max()).sum():>3} "
          f"intervals   |   min ${a.price.min():>9,.2f} on "
          f"{(a.price == a.price.min()).sum():>3} intervals")
print("     Cap was $738 through 2024 (binding, last hit 2024-12-11), $1,100 from 2025.")
print("     The 2025 and 2026 maxima are unique values under a higher ceiling, not a cap.")

print("\n3. MANUAL OVERRIDES — material to the count, immaterial to the mean:")
for y in (2025, 2026):
    a = d[d.year == y]
    b = a[a.failure_reason != "MarketAnalystOverride"]
    print(f"     {y}:  all ${a.price.mean():.2f} (n={len(a):,})   "
          f"ex-override ${b.price.mean():.2f} (n={len(b):,})   "
          f"delta ${a.price.mean() - b.price.mean():+.2f}")
print("     Too small to move any conclusion here, so the base series is kept and flagged.")

1. LIKE-FOR-LIKE (January–August only) — the rise survives:
          n   mean  median    std
year                             
2024  11712   82.0    76.5  134.0
2025  11664   93.7    90.3   57.7
2026  11664  109.0   102.5   45.3

   full-year vs Jan–Aug mean, for the years where both exist:
     2024:  full $79.3   Jan–Aug $82.0
     2025:  full $87.7   Jan–Aug $93.7

2. CAP AND FLOOR — 'times at the max' is partly a rule change; the floor is not:
     2023:  max $  738.00 on  43 intervals   |   min $  -716.71 on   2 intervals
     2024:  max $  738.00 on 123 intervals   |   min $-1,000.00 on  44 intervals
     2025:  max $1,100.00 on   1 intervals   |   min $  -150.52 on   1 intervals
     2026:  max $1,000.00 on   1 intervals   |   min $   -16.72 on   1 intervals
     Cap was $738 through 2024 (binding, last hit 2024-12-11), $1,100 from 2025.
     The 2025 and 2026 maxima are unique values under a higher ceiling, not a cap.

3. MANUAL OVERRIDES — material to the count, immaterial to

## What this notebook establishes

- **Volatility collapsed.** Between 2024 and 2026 the rolling 10th–90th band narrows by 64%, the
  standard deviation by 68%, and the median daily spread by 84% — 65–85% depending on the measure.
  The fall runs *through* the January 2025 cap change rather than stepping at it.
- **The level rose anyway** — mean $82 -> $94 -> $109 on a like-for-like January–August basis. Prices
  went up and steadied at the same time, which is not the usual pairing.
- **Negative prices went from routine to rare**, 22% of all 2024 intervals to 0.4% in 2026, and the
  loss is concentrated exactly where rooftop solar put them, in the middle of the day.
- **The daily shape flattened from both ends, rather than shifting down.** Mean 18:00 price fell from
  $204 to $137 while the overnight mean *rose* from $70 to $102. Peak and trough are converging,
  which is what price-taking storage does to a market.

The open question this hands to the battery notebook is whether the storage fleet caused the
flattening or merely arrived alongside it — and, if it did, what that implies for the arbitrage
revenue the next tranche of batteries is being built against.
